# 🛒 AI-Based Market Basket Analysis for Retail Promotion and Product Placement
### CP020003 Artificial Intelligence — Final Project (KKU 2026)

**เป้าหมาย:** หาว่าสินค้าอะไรมักถูกซื้อด้วยกัน แล้วแปลงผลเป็น 2 อย่างที่ร้านเอาไปใช้ได้จริง
1. **Promotion:** จัดชุดโปรโมชัน (bundle / ซื้อ A ลด B) พร้อมประเมินรายได้ที่อาจเพิ่มขึ้น
2. **Product Placement:** จัดกลุ่มสินค้าที่ควรวางใกล้กันเป็น "โซน"

**สิ่งที่ต่างจากแล็บ Week 7:** เราไม่ได้แค่หากฎ แต่**พิสูจน์ว่ากฎใช้ได้กับอนาคต** โดยหากฎจากข้อมูลช่วงแรก แล้วทดสอบกับช่วงหลัง (แบ่งตามเวลา ไม่มี leakage) และเทียบกับ baseline

| ส่วน | หัวข้อ (ตรงกับโครงนำเสนอของอาจารย์) |
|---|---|
| 0–1 | Setup + ดึงข้อมูลลง Google Drive |
| 2–3 | Dataset & EDA + ทำความสะอาด |
| 4–5 | แบ่ง train/test ตามเวลา + สร้าง basket |
| 6–8 | Methodology: Apriori, FP-Growth, Item2Vec |
| 9 | Evaluation: Hit@K เทียบ baseline |
| 10–11 | Impact: Promotion + Product Placement |
| 12 | สรุปและข้อจำกัด |

## 0. Setup
รันครั้งเดียวต่อ session ของ Colab

In [ ]:
!pip -q install mlxtend gensim

In [ ]:
import warnings, os, io, time, zipfile
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import requests

from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
from mlxtend.preprocessing import TransactionEncoder
from gensim.models import Word2Vec

SEED = 42
np.random.seed(SEED)
sns.set_style("whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 60)
print("พร้อมแล้ว ✅")

## 1. ดึงข้อมูลลง Google Drive แล้วโหลดจาก Drive 📥

ขั้นตอน:
1. Mount Google Drive (Colab จะขอสิทธิ์ ให้กด Allow)
2. สร้างโฟลเดอร์ `MyDrive/CP_Project/data`
3. ถ้ายังไม่มีไฟล์ใน Drive จะดาวน์โหลด dataset จาก repo ของอาจารย์ (`KKU-EUOnlineRetail.csv.zip`) มาเก็บไว้ใน Drive ให้ครั้งเดียว ครั้งต่อไปจะโหลดจาก Drive เลย
4. ถ้าอยากใช้ไฟล์ `data.csv` ของตัวเอง ให้อัปโหลดไปไว้ที่ `MyDrive/CP_Project/data/data.csv` แล้วโค้ดจะใช้ไฟล์นั้นก่อน

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/CP_Project"
except ImportError:
    BASE_DIR = "./CP_Project"   # ถ้ารันนอก Colab จะเก็บไว้ในเครื่อง

DATA_DIR = os.path.join(BASE_DIR, "data")
OUT_DIR = os.path.join(BASE_DIR, "outputs")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

DATA_URL = ("https://github.com/kaopanboonyuen/CP020003_ArtificialIntelligence_2026s1"
            "/raw/main/dataset/KKU-EUOnlineRetail.csv.zip")
ZIP_PATH = os.path.join(DATA_DIR, "KKU-EUOnlineRetail.csv.zip")
OWN_CSV = os.path.join(DATA_DIR, "data.csv")

if os.path.exists(OWN_CSV):
    print("ใช้ไฟล์ของเราเองใน Drive:", OWN_CSV)
    CSV_SOURCE = OWN_CSV
else:
    if not os.path.exists(ZIP_PATH):
        print("ยังไม่มีไฟล์ใน Drive กำลังดาวน์โหลด...")
        r = requests.get(DATA_URL, timeout=120)
        r.raise_for_status()
        with open(ZIP_PATH, "wb") as f:
            f.write(r.content)
        print(f"บันทึกลง Drive แล้ว: {ZIP_PATH} ({len(r.content)/1e6:.1f} MB)")
    else:
        print("มีไฟล์ใน Drive แล้ว โหลดจาก Drive:", ZIP_PATH)
    CSV_SOURCE = ZIP_PATH

In [ ]:
def load_retail(path):
    # ไฟล์นี้มีเครื่องหมาย £ ต้องอ่านด้วย latin-1 (ISO-8859-1) ไม่งั้น error
    dtypes = {"InvoiceNo": str, "StockCode": str, "CustomerID": str}
    if path.endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            name = [n for n in z.namelist() if n.lower().endswith(".csv") and not n.startswith("__MACOSX")][0]
            with z.open(name) as f:
                return pd.read_csv(f, encoding="ISO-8859-1", dtype=dtypes)
    return pd.read_csv(path, encoding="ISO-8859-1", dtype=dtypes)

retail = load_retail(CSV_SOURCE)
retail["InvoiceDate"] = pd.to_datetime(retail["InvoiceDate"], format="%m/%d/%Y %H:%M")
print("Shape:", retail.shape)
retail.head()

## 2. Dataset & EDA 🔍
ข้อมูลการซื้อขายของร้านขายของขวัญออนไลน์ในอังกฤษ ช่วง ธ.ค. 2010 – ธ.ค. 2011 หนึ่งแถว = สินค้าหนึ่งรายการในใบเสร็จ

| คอลัมน์ | ความหมาย |
|---|---|
| `InvoiceNo` | เลขใบเสร็จ (ขึ้นต้นด้วย `C` = ยกเลิก) |
| `StockCode` / `Description` | รหัสและชื่อสินค้า |
| `Quantity` | จำนวน (ติดลบ = คืนสินค้า) |
| `InvoiceDate` | วันเวลาที่ซื้อ |
| `UnitPrice` | ราคาต่อชิ้น (ปอนด์) |
| `CustomerID` | รหัสลูกค้า (ว่างได้ถ้าไม่ได้ล็อกอิน) |
| `Country` | ประเทศลูกค้า |

In [ ]:
print("ช่วงเวลา:", retail["InvoiceDate"].min(), "→", retail["InvoiceDate"].max())
print("ใบเสร็จ:", retail["InvoiceNo"].nunique(), "| สินค้า:", retail["StockCode"].nunique(),
      "| ลูกค้า:", retail["CustomerID"].nunique(), "| ประเทศ:", retail["Country"].nunique())
print()
quality = pd.Series({
    "CustomerID ว่าง": retail["CustomerID"].isna().sum(),
    "Description ว่าง": retail["Description"].isna().sum(),
    "แถวซ้ำทั้งแถว": retail.duplicated().sum(),
    "ใบเสร็จยกเลิก (C)": retail["InvoiceNo"].str.startswith("C").sum(),
    "Quantity <= 0": (retail["Quantity"] <= 0).sum(),
    "UnitPrice <= 0": (retail["UnitPrice"] <= 0).sum(),
})
quality.to_frame("จำนวนแถว")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
retail["Country"].value_counts().head(10).plot.barh(ax=axes[0], color="#4C72B0")
axes[0].invert_yaxis(); axes[0].set_title("Top 10 ประเทศ (จำนวนแถว)")

monthly = retail.set_index("InvoiceDate").resample("MS")["InvoiceNo"].nunique()
monthly.plot(ax=axes[1], marker="o", color="#DD8452"); axes[1].set_title("จำนวนใบเสร็จต่อเดือน")

items_per_basket = retail.groupby("InvoiceNo")["StockCode"].nunique()
items_per_basket.clip(upper=60).hist(bins=60, ax=axes[2], color="#55A868")
axes[2].set_title("จำนวนสินค้าต่อใบเสร็จ (ตัดที่ 60)")
plt.tight_layout(); plt.show()
print("สินค้าต่อใบเสร็จ: median =", items_per_basket.median())

> 💡 **สังเกต:** ข้อมูลกว่า 90% มาจาก UK และยอดขายพุ่งช่วง ก.ย.–พ.ย. (ช่วงเทศกาล) แปลว่าพฤติกรรมช่วงท้ายปีอาจต่างจากต้นปี ซึ่งเป็นเหตุผลที่เราต้องทดสอบกฎกับข้อมูลช่วงเวลาที่โมเดลไม่เคยเห็น

## 3. ทำความสะอาดข้อมูล 🧹
1. ลบแถวซ้ำทั้งแถว
2. ลบใบเสร็จที่ยกเลิก, `Quantity <= 0` และ `UnitPrice <= 0`
3. ลบแถวที่ไม่มี `Description` และตัดช่องว่างหัวท้าย
4. ลบรายการที่ไม่ใช่สินค้า เช่น ค่าส่ง (POSTAGE), ค่าธรรมเนียม, การปรับยอด
5. ใช้ `StockCode` เป็นตัวระบุสินค้า และเก็บชื่อที่พบบ่อยที่สุดไว้แสดงผล

ไม่ลบแถวที่ไม่มี `CustomerID` เพราะ market basket ดูตามใบเสร็จ ไม่ต้องรู้ว่าเป็นใคร

In [ ]:
NON_PRODUCT = {"POST", "DOT", "M", "D", "C2", "S", "B", "CRUK", "PADS", "BANK CHARGES",
               "AMAZONFEE", "ADJUST", "ADJUST2", "TEST001", "TEST002"}

clean = retail.drop_duplicates().copy()
clean = clean[~clean["InvoiceNo"].str.startswith("C")]
clean = clean[(clean["Quantity"] > 0) & (clean["UnitPrice"] > 0)]
clean = clean.dropna(subset=["Description"])
clean["Description"] = clean["Description"].str.strip()
clean["StockCode"] = clean["StockCode"].str.upper().str.strip()
clean = clean[~clean["StockCode"].isin(NON_PRODUCT)]
clean = clean[clean["StockCode"].str.contains(r"\d", regex=True)]   # รหัสสินค้าจริงมีตัวเลข
clean["Revenue"] = clean["Quantity"] * clean["UnitPrice"]

# ชื่อสินค้าที่ใช้แสดงผล: ชื่อที่พบบ่อยที่สุดของแต่ละ StockCode
name_of = clean.groupby("StockCode")["Description"].agg(lambda s: s.value_counts().index[0])
price_of = clean.groupby("StockCode")["UnitPrice"].median()

print(f"ก่อนทำความสะอาด: {len(retail):,} แถว")
print(f"หลังทำความสะอาด: {len(clean):,} แถว ({len(clean)/len(retail):.1%})")
print(f"สินค้า: {clean['StockCode'].nunique():,} | ใบเสร็จ: {clean['InvoiceNo'].nunique():,}")

## 4. เลือกตลาด + แบ่ง train/test ตามเวลา ⏱️

- **ตลาด:** ใช้ UK เพราะข้อมูลเยอะที่สุด และพฤติกรรมต่างประเทศไม่ควรเอามาปนกัน (เปลี่ยน `COUNTRY` ได้)
- **แบ่งตามเวลา:** หากฎจากข้อมูล **ธ.ค. 2010 – ส.ค. 2011 (train)** แล้วทดสอบกับ **ก.ย. – ธ.ค. 2011 (test)**
  - เหตุผล: ในชีวิตจริงร้านจะใช้กฎที่หาจากอดีตไปทำโปรในอนาคต ถ้าสุ่มแบ่ง ใบเสร็จช่วงเดียวกันจะอยู่ทั้งสองฝั่ง ทำให้ผลดูดีเกินจริง
- ตัดใบเสร็จขนาดใหญ่ผิดปกติ (เกิน 99th percentile ของ train) ซึ่งน่าจะเป็นลูกค้าขายส่ง และจะทำให้เกิดกฎปลอม

In [ ]:
COUNTRY = "United Kingdom"
SPLIT_DATE = "2011-09-01"

mkt = clean[clean["Country"] == COUNTRY]
train_rows = mkt[mkt["InvoiceDate"] < SPLIT_DATE]
test_rows = mkt[mkt["InvoiceDate"] >= SPLIT_DATE]

def to_baskets(rows):
    return rows.groupby("InvoiceNo")["StockCode"].apply(lambda s: sorted(set(s)))

train_baskets = to_baskets(train_rows)
test_baskets = to_baskets(test_rows)

MAX_ITEMS = int(train_baskets.apply(len).quantile(0.99))   # คำนวณจาก train เท่านั้น
train_baskets = train_baskets[train_baskets.apply(len).between(1, MAX_ITEMS)]
test_baskets = test_baskets[test_baskets.apply(len).between(1, MAX_ITEMS)]

print(f"ตัดใบเสร็จที่มีสินค้าเกิน {MAX_ITEMS} รายการ")
print(f"Train: {len(train_baskets):,} ใบเสร็จ ({train_rows['InvoiceDate'].min():%Y-%m-%d} → {train_rows['InvoiceDate'].max():%Y-%m-%d})")
print(f"Test : {len(test_baskets):,} ใบเสร็จ ({test_rows['InvoiceDate'].min():%Y-%m-%d} → {test_rows['InvoiceDate'].max():%Y-%m-%d})")

## 5. แปลงเป็น basket แบบ one-hot 🧺
แต่ละแถว = ใบเสร็จ 1 ใบ, แต่ละคอลัมน์ = สินค้า 1 ชิ้น, ค่า True = มีสินค้านั้นในใบเสร็จ (สนใจแค่มีหรือไม่มี ไม่สนจำนวน)

In [ ]:
te = TransactionEncoder()
basket_train = pd.DataFrame(te.fit(train_baskets).transform(train_baskets), columns=te.columns_)
print("Basket matrix:", basket_train.shape, "| ความหนาแน่น:", round(basket_train.values.mean(), 5))

item_support = basket_train.mean().sort_values(ascending=False)
ax = item_support.reset_index(drop=True).plot(logy=True, figsize=(8, 3.5))
for s in [0.01, 0.02, 0.03]:
    ax.axhline(s, ls="--", lw=1, color="gray"); ax.text(len(item_support)*0.8, s*1.1, f"support {s}")
ax.set_title("Long tail: support ของสินค้าแต่ละตัว (เรียงจากมากไปน้อย)"); ax.set_xlabel("อันดับสินค้า")
plt.show()
for s in [0.01, 0.02, 0.03]:
    print(f"min_support={s}: มีสินค้าผ่านเกณฑ์ {(item_support >= s).sum():,} ตัว จาก {len(item_support):,}")

> 💡 **เลือก `min_support = 0.01`** (ประมาณ 1% ของใบเสร็จ หรือ ~100+ ใบ) เพราะสูงพอให้กฎมีตัวอย่างรองรับ แต่ยังครอบคลุมสินค้าหลายร้อยตัว ส่วนสินค้าใน long tail ที่ไม่ผ่านเกณฑ์ เราจะใช้ Item2Vec ช่วยในส่วนที่ 8

## 6. Technique 1: Apriori 🌲
ใช้หลัก "ถ้าชุดเล็กไม่บ่อย ชุดใหญ่ที่มีชุดเล็กนั้นก็ไม่บ่อยแน่นอน" เพื่อตัดชุดที่ไม่ต้องคำนวณทิ้ง

In [ ]:
MIN_SUPPORT = 0.01
# ส่งเฉพาะสินค้าที่ผ่าน min_support ให้ Apriori เพื่อประหยัดหน่วยความจำ (ผลเหมือนเดิมตามหลัก downward closure)
frequent_cols = item_support[item_support >= MIN_SUPPORT].index
basket_small = basket_train[frequent_cols]

t0 = time.time()
fi_apriori = apriori(basket_small, min_support=MIN_SUPPORT, use_colnames=True, max_len=3)
apriori_time = time.time() - t0
print(f"Apriori: {len(fi_apriori):,} frequent itemsets ใน {apriori_time:.2f}s")

## 7. Technique 2: FP-Growth 🌳
บีบข้อมูลเป็น FP-tree แล้วหา itemset จากต้นไม้โดยตรง ไม่ต้องสร้าง candidate ควรได้ผลเหมือน Apriori ทุกตัว แต่เร็วกว่า

In [ ]:
t0 = time.time()
fi_fp = fpgrowth(basket_train, min_support=MIN_SUPPORT, use_colnames=True, max_len=3)
fp_time = time.time() - t0
print(f"FP-Growth: {len(fi_fp):,} frequent itemsets ใน {fp_time:.2f}s (บนสินค้าทั้งหมด {basket_train.shape[1]:,} ตัว)")
print("ได้ itemset ชุดเดียวกับ Apriori ไหม?", set(fi_apriori["itemsets"]) == set(fi_fp["itemsets"]))

In [ ]:
rules = association_rules(fi_fp, metric="lift", min_threshold=1.0)
MIN_CONF, MIN_LIFT = 0.3, 3.0
strong = rules[(rules["confidence"] >= MIN_CONF) & (rules["lift"] >= MIN_LIFT)].copy()

def names(itemset):
    return " + ".join(name_of.get(i, i) for i in sorted(itemset))
strong["IF (ซื้อ)"] = strong["antecedents"].apply(names)
strong["THEN (มักซื้อ)"] = strong["consequents"].apply(names)
strong = strong.sort_values(["lift", "confidence"], ascending=False)

print(f"กฎทั้งหมด (lift>1): {len(rules):,} | กฎที่ผ่านเกณฑ์ confidence>={MIN_CONF}, lift>={MIN_LIFT}: {len(strong):,}")
strong[["IF (ซื้อ)", "THEN (มักซื้อ)", "support", "confidence", "lift"]].head(15).round(3)

In [ ]:
plt.figure(figsize=(8, 5))
sc = plt.scatter(rules["support"], rules["confidence"], c=rules["lift"], cmap="viridis", s=12, alpha=0.6)
plt.colorbar(sc, label="lift")
plt.axhline(MIN_CONF, ls="--", color="red", lw=1)
plt.xlabel("support"); plt.ylabel("confidence"); plt.title("กฎทั้งหมด: support vs confidence (สีคือ lift)")
plt.show()

## 8. Technique 3: Item2Vec (Deep-learning style embedding) 🧠
มองใบเสร็จเป็น "ประโยค" และสินค้าเป็น "คำ" แล้วฝึก Word2Vec ให้สินค้าที่มักอยู่ใบเดียวกันมี vector ใกล้กัน
ข้อดีคือใช้กับสินค้าใน long tail ที่ support ต่ำเกินกว่าจะเกิดกฎได้

In [ ]:
sentences = [b for b in train_baskets if len(b) >= 2]
w2v = Word2Vec(sentences, vector_size=64, window=max(len(b) for b in sentences), min_count=3,
               sg=1, negative=10, epochs=15, workers=4, seed=SEED)
print("สินค้าที่มี embedding:", len(w2v.wv), "ตัว | สินค้าที่อยู่ในกฎ:",
      len(set().union(*rules["antecedents"], *rules["consequents"])), "ตัว")

probe = item_support.index[0]
print(f"\nสินค้าที่คล้าย '{name_of[probe]}':")
for code_, sim in w2v.wv.most_similar(probe, topn=5):
    print(f"  {sim:.2f}  {name_of.get(code_, code_)}")

## 9. Evaluation: กฎใช้ได้กับอนาคตจริงไหม? 📏

**วิธีทดสอบ (leave-one-out บน test set):** ในใบเสร็จช่วง ก.ย.–ธ.ค. ที่มีสินค้า ≥ 2 ชิ้น สุ่มซ่อนสินค้า 1 ชิ้น แล้วให้แต่ละวิธีแนะนำ K ชิ้นจากสินค้าที่เหลือในตะกร้า ถ้าสินค้าที่ซ่อนอยู่ในลิสต์ถือว่า "hit"

**Metric:** `Hit@K` = สัดส่วนใบเสร็จที่ทายถูก, `Coverage` = สัดส่วนใบเสร็จที่วิธีนั้นแนะนำอะไรได้

| วิธี | ใช้แนะนำอย่างไร |
|---|---|
| Baseline: Popularity | แนะนำสินค้าขายดีสุดที่ยังไม่อยู่ในตะกร้า (ไม่ใช้ AI) |
| Association Rules | ใช้กฎที่ antecedent อยู่ในตะกร้า เรียงตาม confidence × lift |
| Item2Vec | หาสินค้าที่ vector ใกล้ค่าเฉลี่ยของสินค้าในตะกร้าที่สุด |
| Hybrid | ใช้กฎก่อน ถ้าไม่พอค่อยเติมด้วย Item2Vec |

In [ ]:
K = 10
rng = np.random.default_rng(SEED)
popular = list(item_support.index)

rule_list = [(set(a), list(c), conf * lift) for a, c, conf, lift in
             zip(rules["antecedents"], rules["consequents"], rules["confidence"], rules["lift"])]

def rec_popular(basket, k=K):
    return [i for i in popular if i not in basket][:k]

def rec_rules(basket, k=K):
    scores = {}
    for ante, cons, score in rule_list:
        if ante <= basket:
            for c in cons:
                if c not in basket:
                    scores[c] = max(scores.get(c, 0), score)
    return [i for i, _ in sorted(scores.items(), key=lambda x: -x[1])][:k]

def rec_item2vec(basket, k=K):
    known = [i for i in basket if i in w2v.wv]
    if not known:
        return []
    sims = w2v.wv.most_similar(positive=known, topn=k + len(basket))
    return [i for i, _ in sims if i not in basket][:k]

def rec_hybrid(basket, k=K):
    out = rec_rules(basket, k)
    for i in rec_item2vec(basket, k * 2):
        if len(out) >= k: break
        if i not in out: out.append(i)
    return out

eval_baskets = [b for b in test_baskets if len(b) >= 2]
eval_baskets = [eval_baskets[i] for i in rng.choice(len(eval_baskets), size=min(3000, len(eval_baskets)), replace=False)]
cases = []
for b in eval_baskets:
    hidden = b[rng.integers(len(b))]
    cases.append((set(b) - {hidden}, hidden))

methods = {"Baseline: Popularity": rec_popular, "Association Rules": rec_rules,
           "Item2Vec": rec_item2vec, "Hybrid (Rules + Item2Vec)": rec_hybrid}
results = []
for name, fn in methods.items():
    hits, covered = 0, 0
    for basket, hidden in cases:
        recs = fn(basket)
        covered += bool(recs)
        hits += hidden in recs
    results.append({"Method": name, f"Hit@{K}": hits / len(cases), "Coverage": covered / len(cases)})

leaderboard = pd.DataFrame(results).set_index("Method").sort_values(f"Hit@{K}", ascending=False)
print(f"ทดสอบบน {len(cases):,} ใบเสร็จจากช่วง test (โมเดลไม่เคยเห็น)")
leaderboard.round(3)

In [ ]:
ax = leaderboard[f"Hit@{K}"].iloc[::-1].plot.barh(figsize=(8, 3.5),
        color=["#94a3b8" if "Baseline" in m else "#2563eb" for m in leaderboard.index[::-1]])
ax.set_title(f"Hit@{K} บน test set (ก.ย.–ธ.ค. 2011)"); ax.set_xlabel(f"Hit@{K}")
for i, v in enumerate(leaderboard[f"Hit@{K}"].iloc[::-1]):
    ax.text(v, i, f" {v:.1%}", va="center")
plt.tight_layout(); plt.show()

In [ ]:
# กฎยังจริงอยู่ไหมในช่วง test? คำนวณ support/confidence/lift ของกฎเดิมใหม่บนข้อมูล test
known_items = set(te.columns_)   # สินค้าใหม่ที่ไม่เคยเห็นใน train ไม่มีในกฎอยู่แล้ว จึงตัดออก
test_known = [[i for i in b if i in known_items] for b in test_baskets]
basket_test = pd.DataFrame(te.transform(test_known), columns=te.columns_)

def rule_stats(ante, cons, B):
    a = B[list(ante)].all(axis=1); c = B[list(cons)].all(axis=1)
    sup = (a & c).mean(); conf = sup / a.mean() if a.mean() > 0 else np.nan
    return pd.Series({"test_confidence": conf, "test_lift": conf / c.mean() if c.mean() > 0 else np.nan})

check = strong.head(30).copy()
check[["test_confidence", "test_lift"]] = check.apply(lambda r: rule_stats(r["antecedents"], r["consequents"], basket_test), axis=1)
print(f"กฎแข็งแรง 30 อันดับแรก: ยังมี lift > 1 ในช่วง test {(check['test_lift'] > 1).mean():.0%}")
check[["IF (ซื้อ)", "THEN (มักซื้อ)", "confidence", "test_confidence", "lift", "test_lift"]].round(2).head(10)

## 10. Impact #1: Promotion 💸
เลือกกฎสำหรับทำโปรโมชัน **ซื้อ A ลด B** และประเมินรายได้ที่อาจเพิ่มขึ้นจากข้อมูลช่วง test

**สมมติฐาน (ต้องบอกตอนนำเสนอ):** ในใบเสร็จที่มี A แต่ไม่มี B ถ้าเสนอโปรแล้วมีลูกค้า `CONVERSION` ส่วนหนึ่งซื้อ B เพิ่ม ด้วยส่วนลด `DISCOUNT`

In [ ]:
CONVERSION, DISCOUNT = 0.10, 0.10
single = strong[(strong["antecedents"].apply(len) <= 2) & (strong["consequents"].apply(len) == 1)].copy()

def promo_value(r):
    a = basket_test[list(r["antecedents"])].all(axis=1)
    b = basket_test[list(r["consequents"])[0]]
    opportunity = int((a & ~b).sum())
    price_b = price_of[list(r["consequents"])[0]]
    return pd.Series({"ใบเสร็จที่มี A แต่ไม่มี B (test)": opportunity,
                      "ราคา B (£)": price_b,
                      "รายได้เพิ่มโดยประมาณ (£)": opportunity * CONVERSION * price_b * (1 - DISCOUNT)})

promo = pd.concat([single, single.apply(promo_value, axis=1)], axis=1)
promo = promo.sort_values("รายได้เพิ่มโดยประมาณ (£)", ascending=False).head(10)
promo[["IF (ซื้อ)", "THEN (มักซื้อ)", "confidence", "lift",
       "ใบเสร็จที่มี A แต่ไม่มี B (test)", "ราคา B (£)", "รายได้เพิ่มโดยประมาณ (£)"]].round(2)

## 11. Impact #2: Product Placement 🗺️
สร้างกราฟ: จุด = สินค้า, เส้น = มีกฎแข็งแรงเชื่อมกัน แล้วใช้ community detection แบ่งเป็น **โซน** สินค้าในโซนเดียวกันควรวางใกล้กัน (หน้าร้าน) หรืออยู่หน้าเดียวกัน / "Frequently bought together" (ออนไลน์)

In [ ]:
G = nx.Graph()
for _, r in strong.iterrows():
    for a in r["antecedents"]:
        for c in r["consequents"]:
            w = max(G[a][c]["weight"], r["lift"]) if G.has_edge(a, c) else r["lift"]
            G.add_edge(a, c, weight=w)

communities = sorted(nx.algorithms.community.greedy_modularity_communities(G, weight="weight"), key=len, reverse=True)
zones = []
for z, comm in enumerate(communities, 1):
    ranked = sorted(comm, key=lambda i: -item_support[i])
    zones.append({"โซน": z, "จำนวนสินค้า": len(comm),
                  "ตัวอย่างสินค้า": " | ".join(name_of[i] for i in ranked[:4])})
zones_df = pd.DataFrame(zones).set_index("โซน")
print(f"สินค้า {G.number_of_nodes()} ตัว แบ่งได้ {len(communities)} โซน")
zones_df.head(10)

In [ ]:
top_zones = communities[:6]
H = G.subgraph(set().union(*top_zones))
color_of = {n: z for z, comm in enumerate(top_zones) for n in comm}
pos = nx.spring_layout(H, seed=SEED, k=0.35)
plt.figure(figsize=(12, 9))
nx.draw_networkx_edges(H, pos, alpha=0.25)
nx.draw_networkx_nodes(H, pos, node_color=[color_of[n] for n in H.nodes], cmap="tab10",
                       node_size=[3000 * item_support[n] + 30 for n in H.nodes])
hubs = sorted(H.nodes, key=lambda n: -H.degree(n))[:15]
nx.draw_networkx_labels(H, pos, labels={n: name_of[n][:22] for n in hubs}, font_size=7)
plt.title("โซนสินค้า 6 โซนใหญ่ (สี = โซน, ขนาด = ความนิยม)"); plt.axis("off"); plt.show()

In [ ]:
# บันทึกผลลัพธ์ลง Drive ไว้ใช้ทำสไลด์
strong[["IF (ซื้อ)", "THEN (มักซื้อ)", "support", "confidence", "lift"]].to_csv(os.path.join(OUT_DIR, "rules.csv"), index=False)
promo.drop(columns=["antecedents", "consequents"]).to_csv(os.path.join(OUT_DIR, "promotions.csv"), index=False)
zones_df.to_csv(os.path.join(OUT_DIR, "zones.csv"))
leaderboard.to_csv(os.path.join(OUT_DIR, "leaderboard.csv"))
print("บันทึกไว้ที่:", OUT_DIR, os.listdir(OUT_DIR))

## 12. สรุปและข้อจำกัด 📝

**ผลหลัก** (ดูตัวเลขจริงจากเซลล์ด้านบน)
- Apriori และ FP-Growth ได้ itemset ชุดเดียวกัน แต่ FP-Growth เร็วกว่าและรันบนสินค้าทั้งหมดได้
- ทุกวิธีที่เป็น AI ทายสินค้าที่ลูกค้าจะซื้อในอนาคตได้ดีกว่า baseline ที่แนะนำแค่สินค้าขายดี
- กฎส่วนใหญ่ยังใช้ได้ในช่วง test แปลว่าเอาไปทำโปรได้จริง ไม่ใช่แค่จำข้อมูลเก่า

**ข้อจำกัดของข้อมูล** (เกณฑ์ Dataset Understanding)
- ข้อมูลปี 2010–2011 จากร้านเดียวในอังกฤษ ขายของขวัญเป็นหลัก อาจใช้กับร้านค้าปลีกไทยไม่ได้ตรงๆ
- ลูกค้าส่วนหนึ่งเป็นผู้ค้าส่ง ทำให้ตะกร้าใหญ่ผิดปกติ (เราตัดที่ 99th percentile)
- ช่วง test เป็นช่วงเทศกาล พฤติกรรมอาจต่างจากช่วงปกติ
- รายได้ที่ประเมินขึ้นอยู่กับสมมติฐาน conversion และส่วนลด ควรยืนยันด้วย A/B test จริง

**ต่อยอด**
- ทำเว็บเดโม: เลือกสินค้าในตะกร้า แล้วระบบแนะนำสินค้าที่ควรซื้อคู่กัน (Gradio / Streamlit)
- แยกกฎตามฤดูกาลหรือกลุ่มลูกค้า (ผสมกับ customer segmentation จาก Week 5)